# Bayesian Hyperparameter Space Optimization and Automated Model Auditing Pipeline

In [1]:
%load_ext autoreload
%autoreload 2

import os
import mlflow
import optuna
from optuna.integration.mlflow import MLflowCallback
from optuna.pruners import MedianPruner
import numpy as np
import pandas as pd

from src import optuna_optimization as utils

/Users/hector.vargas/repos/ml_hands_on_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Path Configuration & MLflow Backend Binding

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 5
EXPERIMENT_NAME = "customer-churn-optuna"

ROOT_DIR = os.path.abspath("../")
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")

# Set the tracking URI immediately to lock it to SQLite
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

# Explicitly create the experiment with your designated mlartifacts folder path
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    mlflow.create_experiment(
        name=EXPERIMENT_NAME,
        artifact_location=f"file://{ARTIFACTS_DIR}"
    )

# Active the experiment scope
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='file:///Users/hector.vargas/repos/mlartifacts', creation_time=1781720479070, experiment_id='5', last_update_time=1781720479070, lifecycle_stage='active', name='customer-churn-optuna_rf', tags={}, trace_location=None, workspace='default'>

## 2. Custom Source Code Ingestion

In [3]:
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")
y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()

In [4]:
# Mapping Runtime Feature Schemas
nomod_columns = ['HasCrCard', 'IsActiveMember']
dummyfy_columns = ['Card Type', 'NumOfProducts', 'Geography', 'Gender']
norm_std_columns = ['Balance', 'Point Earned', 'CreditScore', 'Age', 'Tenure', 'Satisfaction Score', 'EstimatedSalary']

## 2. Orchestrate and Initialize Search Parameter Studies

In [32]:
# Initialize Integrated MLflow Tracking Integration Callbacks
mlflow_callback = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="average_precision",
    create_experiment=False, # Keeps Optuna from trying to override our custom artifact path
    mlflow_kwargs={"nested": True}
)

study = optuna.create_study(
    study_name="customer-churn-rf-search-v1",
    direction="maximize",
    pruner=MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=0
    ),
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)

/var/folders/bv/50x24wc545x5mclk_t88ryrc0000gn/T/ipykernel_99254/927694775.py:2: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback = MLflowCallback(
[I 2026-06-17 12:24:38,467] A new study created in memory with name: customer-churn-rf-search-v1


In [6]:
FEATURE_REGISTRY_BINARY = {
    # --- Binary Flags ---
    "is_silver": lambda X: X["Card Type"] == 'SILVER',
    "is_germany": lambda X: X["Geography"] == 'Germany',
    "is_spain": lambda X: X["Geography"] == 'Spain',
    "Num_Of_Products_1": lambda X: X["NumOfProducts"] == 1,
    "Num_Of_Products_2": lambda X: X["NumOfProducts"] == 2,
    "Num_Of_Products_3": lambda X: X["NumOfProducts"] == 3,
    "Num_Of_Products_4": lambda X: X["NumOfProducts"] == 4,
}

FEATURE_REGISTRY_CONTINUOUS = {
    # --- Polynomial & Interaction Terms ---
    "Age_x_IsActive": lambda X: X["Age"] * X["IsActiveMember"],

    # --- Financial & Engagement Ratios ---
    "Balance_per_Product": lambda X: X["Balance"] / (X["NumOfProducts"] + 1),
    "CreditScore_per_Age": lambda X: X["CreditScore"] / (X["Age"] + 1),

    # --- Behavioral Cross-Products ---
    "Inactive_x_Balance": lambda X: (1 - X["IsActiveMember"]) * X["Balance"],

    # --- Monetary Accumulations & Non-linear Scaling ---
    "CreditScore_x_Age": lambda X: X["CreditScore"] * X["Age"],

    # --- Temporal Product Densities ---
    "Products_per_Tenure": lambda X: X["NumOfProducts"] / (X["Tenure"] + 1),
}
# Schema Baseline Columns Definitions
nomod_columns = []

dummyfy_columns = [
    'Gender'
    ]

norm_std_columns = [
    'Point Earned',
    'Satisfaction Score', 
    'EstimatedSalary'
    ]

In [7]:
all_engineered_features = list(FEATURE_REGISTRY_BINARY.keys())

EXPERIMENT_REGISTRY = {
    # Experiment 2: 
    "experiment_2": {
        "passthrough": nomod_columns + list(FEATURE_REGISTRY_BINARY.keys()),
        "standard_scale": norm_std_columns + list(FEATURE_REGISTRY_CONTINUOUS.keys()),
        "one_hot_encode": dummyfy_columns
    },
}
# Combine registries purely for execution lookup inside the transformer
FULL_REGISTRY = {**FEATURE_REGISTRY_BINARY, **FEATURE_REGISTRY_CONTINUOUS}
current_layout = EXPERIMENT_REGISTRY['experiment_2']

In [8]:
# Instantiate the cross-validation objective callable
objective_function = utils.ObjectiveCV(
    X=X_train,
    y=y_train,
    FULL_REGISTRY=FULL_REGISTRY,
    current_layout=current_layout,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
)

## 3. Run Optimization Search Workspace Execution Window

In [39]:
with mlflow.start_run(run_name="optuna_search_parent"):
    
    study.optimize(
        objective_function,
        n_trials=500,
        callbacks=[mlflow_callback]
    )

    # Log global best performance summary attributes
    mlflow.log_metric("best_auc", study.best_value)
    mlflow.log_params(study.best_params)

    # Reconstruct and serialize top performing candidate artifacts pipeline
    best_pipeline = utils.build_pipeline(
        study.best_trial, 
        FULL_REGISTRY,
        current_layout,
        random_state=RANDOM_STATE
    )

    # This internally calls artifact logging functions; they will now map straight to mlartifacts/
    utils.evaluate_and_log_best_model(
        best_pipeline=best_pipeline,
        X_train=X_train, y_train=y_train,
        X_test=X_test, y_test=y_test,
        cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns
    )

[I 2026-06-17 12:50:34,455] Trial 1500 pruned. 
[I 2026-06-17 12:50:37,177] Trial 1501 finished with value: 0.687241765575644 and parameters: {'scaler': 'std', 'encoder': 'drop_first', 'model': 'rf', 'rf_n_estimators': 300, 'rf_max_depth': 9, 'rf_min_samples_split': 18, 'rf_min_samples_leaf': 1}. Best is trial 329 with value: 0.687241765575644.
[I 2026-06-17 12:50:40,040] Trial 1502 finished with value: 0.687241765575644 and parameters: {'scaler': 'std', 'encoder': 'drop_first', 'model': 'rf', 'rf_n_estimators': 300, 'rf_max_depth': 9, 'rf_min_samples_split': 18, 'rf_min_samples_leaf': 1}. Best is trial 329 with value: 0.687241765575644.
[I 2026-06-17 12:50:43,068] Trial 1503 pruned. 
[I 2026-06-17 12:50:43,896] Trial 1504 pruned. 
[I 2026-06-17 12:50:44,643] Trial 1505 pruned. 
[I 2026-06-17 12:50:45,429] Trial 1506 finished with value: 0.687241765575644 and parameters: {'scaler': 'std', 'encoder': 'drop_first', 'model': 'rf', 'rf_n_estimators': 300, 'rf_max_depth': 9, 'rf_min_samples

## 4. Display Session Optimization Diagnostics Results

In [40]:
print(f"\nTop Optimization Average Precision Score Target: {study.best_value:.4f}")
print("\nOptimal Parameter Combinations Selected:")
for parameter_key, parameter_value in study.best_params.items():
    print(f" * {parameter_key}: {parameter_value}")


Top Optimization Average Precision Score Target: 0.6872

Optimal Parameter Combinations Selected:
 * scaler: std
 * encoder: drop_first
 * model: rf
 * rf_n_estimators: 300
 * rf_max_depth: 9
 * rf_min_samples_split: 18
 * rf_min_samples_leaf: 1


In [16]:
import pandas as pd
import numpy as np


def suggest_numeric_ranges(
    study,
    top_quantile=0.95,
    boundary_threshold=0.15,
    expansion_factor=0.5,
):
    """
    Suggest new Optuna search ranges for numeric parameters.

    Parameters
    ----------
    study : optuna.study.Study

    top_quantile : float
        Use top X% of trials to infer the optimum region.

    boundary_threshold : float
        Fraction of range considered "close to boundary".

    expansion_factor : float
        How much to expand the range when boundary pressure exists.

    Returns
    -------
    pd.DataFrame
    """

    df = study.trials_dataframe()

    completed = df[df["state"] == "COMPLETE"].copy()

    threshold = completed["value"].quantile(top_quantile)

    top_trials = completed[completed["value"] >= threshold]

    results = []

    for col in completed.columns:

        if not col.startswith("params_"):
            continue

        if not pd.api.types.is_numeric_dtype(completed[col]):
            continue

        all_values = completed[col].dropna()

        if len(all_values) < 5:
            continue

        top_values = top_trials[col].dropna()

        current_min = all_values.min()
        current_max = all_values.max()

        span = current_max - current_min

        if span == 0:
            continue

        median_top = top_values.median()

        relative_position = (
            median_top - current_min
        ) / span

        if relative_position < boundary_threshold:

            action = "move_left"

            suggested_min = current_min - expansion_factor * span
            suggested_max = current_max

        elif relative_position > (1 - boundary_threshold):

            action = "move_right"

            suggested_min = current_min
            suggested_max = current_max + expansion_factor * span

        else:

            q10 = top_values.quantile(0.10)
            q90 = top_values.quantile(0.90)

            concentration = (q90 - q10) / span

            if concentration < 0.40:

                action = "narrow"

                padding = (q90 - q10) * 0.20

                suggested_min = q10 - padding
                suggested_max = q90 + padding

            else:

                action = "keep"

                suggested_min = current_min
                suggested_max = current_max

        results.append(
            {
                "parameter": col.replace("params_", ""),
                "current_min": current_min,
                "current_max": current_max,
                "best_median": median_top,
                "action": action,
                "suggested_min": suggested_min,
                "suggested_max": suggested_max,
            }
        )

    return (
        pd.DataFrame(results)
        .sort_values(
            ["action", "parameter"],
            ascending=[True, True]
        )
        .reset_index(drop=True)
    )

In [41]:
suggestions = suggest_numeric_ranges(study)

display(suggestions)

,parameter,current_min,current_max,best_median,action,suggested_min,suggested_max
0,rf_min_samples_leaf,1,9,1.0,move_left,-3.0,9.0
1,rf_min_samples_split,4,20,18.0,move_right,4.0,28.0
2,rf_max_depth,2,20,9.0,narrow,9.0,9.0
3,rf_n_estimators,107,498,300.0,narrow,300.0,300.0


In [45]:
import mlflow
import pandas as pd

experiment = mlflow.get_experiment_by_name(
    "customer-churn-optuna"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.average_precision DESC"]
)

runs.head()

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.average_precision,metrics.test_f1,metrics.test_auc,metrics.train_accuracy,...,tags.xgb_reg_lambda_distribution,tags.state,tags.mlflow.source.name,tags.xgb_scale_pos_weight_distribution,tags.xgb_min_child_weight_distribution,tags.xgb_max_depth_distribution,tags.rf_max_depth_distribution,tags.rf_min_samples_split_distribution,tags.rf_min_samples_leaf_distribution,tags.rf_n_estimators_distribution
0,12bfc6e121d3420ab5114f4e24bc51bf,4,FINISHED,file:///Users/hector.vargas/repos/mlartifacts/...,2026-06-17 18:05:35.546000+00:00,2026-06-17 18:05:35.559000+00:00,0.697452,NaN,NaN,NaN,...,"FloatDistribution(high=0.0001, log=True, low=1...",COMPLETE,6.0-havg-optuna-model.ipynb,"FloatDistribution(high=2.2, log=False, low=1.8...",None,None,None,None,None,None
1,8777e06371af4e03ab434a292251d146,4,FINISHED,file:///Users/hector.vargas/repos/mlartifacts/...,2026-06-17 18:06:26.180000+00:00,2026-06-17 18:06:26.193000+00:00,0.697400,NaN,NaN,NaN,...,"FloatDistribution(high=0.0001, log=True, low=1...",COMPLETE,6.0-havg-optuna-model.ipynb,"FloatDistribution(high=2.2, log=False, low=1.8...",None,None,None,None,None,None
2,81c8b62e0fa84e2eb6d53292cfce3bb4,4,FINISHED,file:///Users/hector.vargas/repos/mlartifacts/...,2026-06-17 18:05:37.243000+00:00,2026-06-17 18:05:37.256000+00:00,0.697283,NaN,NaN,NaN,...,"FloatDistribution(high=0.0001, log=True, low=1...",COMPLETE,6.0-havg-optuna-model.ipynb,"FloatDistribution(high=2.2, log=False, low=1.8...",None,None,None,None,None,None
3,1656815e034f43a6a0e67fdd45086a63,4,FINISHED,file:///Users/hector.vargas/repos/mlartifacts/...,2026-06-17 18:05:27.837000+00:00,2026-06-17 18:05:27.850000+00:00,0.696999,NaN,NaN,NaN,...,"FloatDistribution(high=0.0001, log=True, low=1...",COMPLETE,6.0-havg-optuna-model.ipynb,"FloatDistribution(high=2.2, log=False, low=1.8...",None,None,None,None,None,None
4,9aade507b3c44919b76fdc1de0cffab7,4,FINISHED,file:///Users/hector.vargas/repos/mlartifacts/...,2026-06-17 18:07:09.697000+00:00,2026-06-17 18:07:09.710000+00:00,0.696985,NaN,NaN,NaN,...,"FloatDistribution(high=0.0001, log=True, low=1...",COMPLETE,6.0-havg-optuna-model.ipynb,"FloatDistribution(high=2.2, log=False, low=1.8...",None,None,None,None,None,None


In [46]:
best_per_model = (
    runs
    .sort_values("metrics.average_precision", ascending=False)
    .groupby("params.model")
    .head(1)
    .loc[:, [
        "params.model",
        "metrics.average_precision",
        "run_id"
    ]]
)

print(best_per_model)

     params.model  metrics.average_precision                            run_id
0             xgb                   0.697452  12bfc6e121d3420ab5114f4e24bc51bf
2397           rf                   0.685200  c56de9a29b814df289feedb8ea3e73b5


In [47]:
rf_runs = (
    runs[runs["params.model"] == "rf"]
    .sort_values(
        "metrics.average_precision",
        ascending=False
    )
)

rf_runs[
    [
        "run_id",
        "metrics.average_precision",
        "params.rf_max_depth",
        "params.rf_n_estimators",
        "params.rf_min_samples_split",
        "params.rf_min_samples_leaf",
    ]
].head(10)

,run_id,metrics.average_precision,params.rf_max_depth,params.rf_n_estimators,params.rf_min_samples_split,params.rf_min_samples_leaf
2397,c56de9a29b814df289feedb8ea3e73b5,0.685200,10,310,15,1
2628,a16c53dd7d9449dcbaf6ade0de8153af,0.684282,9,441,11,1
2659,125a5f0c59394189b81fc023230f01e2,0.684155,10,494,4,1
2736,e3ed7df927954ac8bc67284f57ca0165,0.683470,8,253,9,1
2738,c31614fe9a544f5584148e60a8dd4d78,0.683451,11,426,18,2
2743,20dcebda9d2c430b908f0dd420114d59,0.683398,10,356,19,1
2787,be8b1776826246dd9561d9c7cf41c100,0.682702,10,473,9,2
2789,405bc109e1d5448cbceb3c28d4e41233,0.682689,13,394,14,2
2792,a1da12a950004950984f74f43fe49a60,0.682669,11,497,6,1
2814,18d8393e596446a983a03221208dc97e,0.682168,8,475,18,2


In [48]:
xgb_runs = (
    runs[runs["params.model"] == "xgb"]
    .sort_values(
        "metrics.average_precision",
        ascending=False
    )
)

xgb_runs[
    [
        "run_id",
        "metrics.average_precision",
        "params.xgb_n_estimators",
        "params.xgb_learning_rate",
        "params.xgb_gamma",
        "params.xgb_scale_pos_weight",
    ]
].head(10)

,run_id,metrics.average_precision,params.xgb_n_estimators,params.xgb_learning_rate,params.xgb_gamma,params.xgb_scale_pos_weight
0,12bfc6e121d3420ab5114f4e24bc51bf,0.697452,865,0.035055556880179084,2.1092220659524137,2.157986273496445
1,8777e06371af4e03ab434a292251d146,0.697400,830,0.03575833510515544,2.0987589756420397,2.173101677356361
2,81c8b62e0fa84e2eb6d53292cfce3bb4,0.697283,877,0.03493911797619159,2.104312098165495,2.1449173001589124
3,1656815e034f43a6a0e67fdd45086a63,0.696999,873,0.03564690739565698,2.202754306282532,2.1890069694481493
4,9aade507b3c44919b76fdc1de0cffab7,0.696985,852,0.03550492781554709,2.1259243465572153,2.1680320981741366
5,5ff0fcc8aa484823b04d79b82619c8c3,0.696959,877,0.03624820049105469,2.0046830766036186,2.1670126672963512
6,719c3c7df3e84360a68cfceba6758be2,0.696934,833,0.036849163883885164,2.173383679697929,2.1762072839755606
7,65368d548ccf474e91e5d0104251118d,0.696933,877,0.03543394638666739,2.1475754301424677,2.177956197786032
8,d044f2a4006c48f69e51510a56bd876e,0.696927,840,0.03565622740574558,2.080521687149425,2.1687380610779443
9,d70cd85b48f349ef929c114c8b91331e,0.696825,877,0.03671937277417893,2.1230654316146915,2.149254716638583


In [49]:
leaderboard = (
    runs[
        [
            "params.model",
            "metrics.average_precision",
            "run_id",
        ]
    ]
    .sort_values(
        "metrics.average_precision",
        ascending=False
    )
)

leaderboard.head(20)

,params.model,metrics.average_precision,run_id
0,xgb,0.697452,12bfc6e121d3420ab5114f4e24bc51bf
1,xgb,0.697400,8777e06371af4e03ab434a292251d146
2,xgb,0.697283,81c8b62e0fa84e2eb6d53292cfce3bb4
3,xgb,0.696999,1656815e034f43a6a0e67fdd45086a63
4,xgb,0.696985,9aade507b3c44919b76fdc1de0cffab7
5,xgb,0.696959,5ff0fcc8aa484823b04d79b82619c8c3
6,xgb,0.696934,719c3c7df3e84360a68cfceba6758be2
7,xgb,0.696933,65368d548ccf474e91e5d0104251118d
8,xgb,0.696927,d044f2a4006c48f69e51510a56bd876e
9,xgb,0.696825,d70cd85b48f349ef929c114c8b91331e
